**1. Just-In-Time Compilation (JIT)**

JAX allows us to transform Python functions by reducing each function into a
sequence of primitive operations, each representing one fundamental unit of com-
putation. Primitives are units of computation; most functions in jax.lax represent
a primitive. JAX utilizes an optimization engine called XLA (Accelerated Linear
Algebra) to compile your Python functions into well optimized machine code. By
simply wrapping a function in jit, JAX merges operations together, minimizes mem-
ory transfers, and runs them incredibly fast.

**JIT compiling a function**

JAX enables operations to execute on CPU/GPU/TPU using the same code. Below
is an example of computing a Scaled Exponential Linear Unit (SELU), an operation
commonly used in deep learning:

In [3]:
import jax
import jax . numpy as jnp
def selu (x , alpha =1.67 , lambda_ =1.05) :
  return lambda_ * jnp . where ( x > 0 , x , alpha * jnp . exp ( x ) - alpha )
x = jnp . arange (1000000)
%timeit selu ( x ) . block_until_ready ()

9.15 ms ± 1.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


The code above is sending one operation at a time to the accelerator. This limits
the ability of the XLA compiler to optimize our functions.
Naturally, what we want to do is give the XLA compiler as much code as possible,
so it can fully optimize it. For this purpose, JAX provides the jax.jit() transforma-
tion, which will JIT compile a JAX-compatible function. Below is an example of
JIT in operation.

In [4]:
selu_jit = jax . jit ( selu )
# Pre - compile the function before timing ...
selu_jit ( x ) . block_until_ready ()
%timeit selu_jit ( x ) . block_until_ready ()

1.41 ms ± 316 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


`selu_jit` was defined as the compiled version of `selu` and called seluj it once on x.
This is where JAX does its tracing, it needs to have some inputs to wrap in tracers,
after all. The `jaxpr` is then compiled using XLA into very efficient code optimized
for your GPU or TPU. Finally, the compiled code is executed to satisfy the call.
We can’t `jit` everthing, common examples are;

* Cannot JIT functions that use Python conditional statements or loops.
* Dynamic Array Shapes
* I/O operations